In [ ]:
import pandas as pd
import requests
import os
import time

from bs4 import BeautifulSoup
from google.colab import files
from tqdm import tqdm

In [ ]:
uploaded = files.upload()

Saving XML Sitemap 13.xml to XML Sitemap 13.xml
Saving XML Sitemap 14.xml to XML Sitemap 14.xml
Saving XML Sitemap 15.xml to XML Sitemap 15.xml


In [ ]:
posts = []

for file in uploaded.keys():

    with open(file, encoding="utf-8") as f:

        soup = BeautifulSoup(f, "xml")

    urls = soup.find_all("url")

    for u in urls:

        posts.append({

            "url": u.loc.text,
            "lastmod": u.lastmod.text

        })

posts = pd.DataFrame(posts)

print(posts.shape)
posts.head()

(2966, 2)


,url,lastmod
0,https://factly.in/telugu-morphed-photo-shared-...,2024-10-01T09:57:22+00:00
1,https://factly.in/the-recent-murder-of-a-woman...,2024-10-01T10:11:33+00:00
2,https://factly.in/youths-beaten-for-bathing-na...,2024-10-01T10:31:20+00:00
3,https://factly.in/telugu-a-video-of-a-clash-be...,2024-10-01T11:07:39+00:00
4,https://factly.in/telugu-al-kabeer-exports-is-...,2024-10-01T12:09:59+00:00


In [ ]:
posts_2025 = posts[
    posts["lastmod"].str.startswith("2025")
].copy()

print("2025 articles:", len(posts_2025))

2025 articles: 1979


In [ ]:
posts_2025 = posts_2025[
    ~posts_2025["url"].str.contains(
        "telugu-|kannada-",
        case=False,
        na=False
    )
]

posts_2025 = posts_2025.drop_duplicates(subset="url")

print("English articles:", len(posts_2025))

English articles: 1237


In [ ]:
import os

checkpoint_file = "factly_2025_checkpoint.csv"

# Resume if checkpoint exists
if os.path.exists(checkpoint_file):

    checkpoint = pd.read_csv(checkpoint_file)

    articles = checkpoint.to_dict("records")

    scraped_urls = set(checkpoint["url"])

    print(f"Resuming from {len(articles)} articles.")

else:

    articles = []

    scraped_urls = set()

    print("Starting from scratch.")

# Only scrape articles that haven't been completed
posts_remaining = posts_2025[
    ~posts_2025["url"].isin(scraped_urls)
].copy()

print("Articles remaining:", len(posts_remaining))

Starting from scratch.
Articles remaining: 1237


In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

for row in tqdm(posts_remaining.itertuples(index=False),
                total=len(posts_remaining)):

    try:

        r = requests.get(
            row.url,
            headers=headers,
            timeout=20
        )

        if r.status_code != 200:
            continue

        soup = BeautifulSoup(r.text, "html.parser")

        # -------------------------
        # Title
        # -------------------------

        title = soup.find("h1")
        title = title.get_text(strip=True) if title else None

        # -------------------------
        # Published Date
        # -------------------------

        meta = soup.find(
            "meta",
            attrs={"property": "article:published_time"}
        )

        if meta:
            date = meta["content"][:10]
        else:
            date = row.lastmod[:10]

        # -------------------------
        # Article Text
        # -------------------------

        paragraphs = soup.find_all("p")

        content = "\n".join(
            p.get_text(" ", strip=True)
            for p in paragraphs
        )

        # -------------------------
        # Save article
        # -------------------------

        articles.append({

            "title": title,
            "url": row.url,
            "date": date,
            "content": content

        })

        # Save checkpoint AFTER EVERY ARTICLE
        pd.DataFrame(articles).to_csv(
            checkpoint_file,
            index=False
        )

        time.sleep(0.5)

    except Exception as e:

        print("\nError:", row.url)
        print(e)

        continue

100%|██████████| 1237/1237 [30:26<00:00,  1.48s/it]


In [ ]:
factly_2025 = pd.DataFrame(articles)

factly_2025.to_csv(
    "factly_2025_english.csv",
    index=False
)

print("Finished!")
print("Articles scraped:", len(factly_2025))

Finished!
Articles scraped: 1237


In [ ]:
!pip -q install google-api-python-client beautifulsoup4 pandas tqdm requests lxml